
# Quantum Neural Network (VQC) Training â€” Diabetes
**SIH 2026 | Egreen Quanta | Diabetes Risk Detection (CDC BRFSS 2015)**

This notebook trains a Variational Quantum Classifier (QNN) on Google Colab.
It also demonstrates **Continuous Learning** (fine-tuning on new hospital data).

## How to use this notebook:
1. Upload `diabetes_train.csv` and `diabetes_test.csv` to Colab using the sidebar.
2. Run all cells in order.
3. Download the trained weights (`diabetes_qnn_weights.json`) at the end.


In [ ]:

!pip install pennylane pennylane-lightning scikit-learn pandas numpy matplotlib seaborn -q


In [ ]:

import os
from google.colab import files

os.makedirs("data", exist_ok=True)
os.makedirs("models", exist_ok=True)

# Upload your CSV files
print("Upload diabetes_train.csv and diabetes_test.csv")
uploaded = files.upload()

for filename in uploaded:
    with open(f"data/{filename}", "wb") as f:
        f.write(uploaded[filename])
    print(f"Saved: data/{filename}")



## Training the Quantum Neural Network â€” Diabetes
The cell below contains the entire QNN architecture adapted for **Diabetes detection**:
- **AngleEmbedding** encodes 8 metabolic/lifestyle indicators into 8 qubits
- **3 Variational Layers** (Reduced for faster iteration)
- **Parameter-Shift Rule** computes exact quantum gradients (no approximations)
- **Neutral Threshold** during training to monitor base accuracy, with automatic threshold search afterward
- **Continuous Learning** demonstrates fine-tuning on 80 new patients

**This is configured for rapid debugging (~15 mins runtime).**


In [ ]:
"""
Quantum Neural Network (VQC) Training Script â€” DIABETES VERSION
================================================================
Designed to run on Google Colab with lightning.qubit C++ backend.
Trains a Variational Quantum Classifier on the CDC BRFSS 2015 Diabetes dataset.
"""
import sys
import time
import numpy as np
import pandas as pd
import pennylane as qml
from pennylane import numpy as pnp
from sklearn.metrics import (accuracy_score, f1_score, classification_report,
                             confusion_matrix, precision_score, recall_score)
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
import json
import os

# ============================================================
# CONFIG â€” Tuned for rapid debugging/iteration
# ============================================================
N_QUBITS = 8             # 8 features: BMI, GenHlth, Age, HighBP, HighChol, Income, PhysHlth, Education
N_LAYERS = 3             # Reduced from 5 to speed up circuit evaluation
LEARNING_RATE = 0.015    # Higher LR to learn faster in fewer epochs
EPOCHS = 40              # Increased to 40 epochs as requested
BATCH_SIZE = 16          # Standard batch size
RANDOM_STATE = 42
RECALL_THRESHOLD = 0.0   # Default threshold during training to monitor unbiased accuracy
CLASS_WEIGHT_POS = 1.0   # Set to 1.0; dataset is already manually balanced 1:1, so extra weight causes bias

np.random.seed(RANDOM_STATE)

# ============================================================
# 1. LOAD AND PREPARE DATA
# ============================================================
print("=" * 60)
print("QUANTUM NEURAL NETWORK (VQC) â€” DIABETES DETECTION")
print("Dataset: CDC BRFSS 2015 (8 metabolic/lifestyle features)")
print("=" * 60)

df_train = pd.read_csv("data/diabetes_train.csv")
df_test = pd.read_csv("data/diabetes_test.csv")

feature_names = [c for c in df_train.columns if c != "target"]
X_train = df_train[feature_names].values
y_train = df_train["target"].values
X_test = df_test[feature_names].values
y_test = df_test["target"].values

# Convert labels: 0 -> -1, 1 -> +1 for QNN

# --- CRITICAL FIX: Scale features to [0, pi] for AngleEmbedding ---
# Standardized features wrap around 2pi randomly, destroying learning.
scaler = MinMaxScaler(feature_range=(0, np.pi))
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

y_train_qnn = 2 * y_train - 1
y_test_qnn = 2 * y_test - 1

# --- Class-Balanced Subsampling ---
# BRFSS is severely imbalanced (84% healthy, 16% diabetic).
# We subsample to create a perfectly balanced training set.
idx_0 = np.where(y_train == 0)[0]  # Healthy
idx_1 = np.where(y_train == 1)[0]  # Diabetic/Prediabetic
np.random.shuffle(idx_0)
np.random.shuffle(idx_1)

# Reduced to 100 per class (200 total) for fast iteration (~15 mins runtime)
n_per_class = 150 # Balanced dataset (300 total) for reasonable 40-epoch runtime

n_minority_available = len(idx_1)
n_majority_available = len(idx_0)
print(f"Available: {n_majority_available} healthy, {n_minority_available} diabetic")

if n_minority_available < n_per_class:
    print(f"WARNING: Only {n_minority_available} diabetic samples. Using all + oversampling.")
    idx_1_sampled = np.concatenate([
        idx_1,
        np.random.choice(idx_1, n_per_class - n_minority_available, replace=True)
    ])
else:
    idx_1_sampled = idx_1[:n_per_class]

idx_0_sampled = idx_0[:n_per_class]
balanced_idx = np.concatenate([idx_0_sampled, idx_1_sampled])
np.random.shuffle(balanced_idx)

X_balanced = X_train[balanced_idx]
y_balanced = y_train_qnn[balanced_idx]

print(f"Training subset: {len(X_balanced)} balanced ({n_per_class} per class)")
print(f"Full training set: {len(X_train)} samples")
print(f"Test set: {len(X_test)} patients")
print(f"Features ({len(feature_names)}): {feature_names}")
print(f"Trainable parameters: {N_LAYERS * N_QUBITS * 3} = {N_LAYERS}L x {N_QUBITS}Q x 3rot")

# ============================================================
# 2. DEFINE THE QUANTUM NEURAL NETWORK (VQC)
# ============================================================
# Try lightning.qubit (C++ accelerated), fall back to default.qubit
try:
    dev = qml.device("lightning.qubit", wires=N_QUBITS)
    print("Using lightning.qubit (C++ accelerated)")
except Exception:
    dev = qml.device("default.qubit", wires=N_QUBITS)
    print("WARNING: lightning.qubit not available, using default.qubit (slower)")

def variational_block(weights, wires):
    """Single variational layer: Rot gates on all qubits + CNOT ring."""
    n_wires = len(wires)
    for i in range(n_wires):
        qml.Rot(weights[i, 0], weights[i, 1], weights[i, 2], wires=wires[i])
    # CNOT ring entanglement
    for i in range(n_wires):
        qml.CNOT(wires=[wires[i], wires[(i + 1) % n_wires]])

@qml.qnode(dev, interface="autograd", diff_method="parameter-shift")
def qnn_circuit(weights, x):
    """8-qubit VQC: AngleEmbedding -> variational layers -> PauliZ measurement."""
    qml.AngleEmbedding(x, wires=range(N_QUBITS))
    for layer in range(N_LAYERS):
        variational_block(weights[layer], wires=range(N_QUBITS))
    return qml.expval(qml.PauliZ(0))

def predict_single(weights, x, threshold=0.0):
    output = float(qnn_circuit(weights, x))
    return 1 if output >= threshold else -1

def predict_batch(weights, X, threshold=0.0):
    return np.array([predict_single(weights, x, threshold) for x in X])

# ============================================================
# 3. COST FUNCTION AND TRAINING LOOP
# ============================================================
def cost(weights, X_batch, y_batch):
    """Weighted MSE loss"""
    predictions = [qnn_circuit(weights, x) for x in X_batch]
    predictions = pnp.stack(predictions)
    errors = (predictions - y_batch) ** 2
    sample_weights = pnp.where(y_batch > 0, CLASS_WEIGHT_POS, 1.0)
    return pnp.mean(errors * sample_weights)

def train_qnn(X_data, y_data, epochs=EPOCHS, lr=LEARNING_RATE, batch_size=BATCH_SIZE, initial_weights=None):

    X_data = pnp.array(X_data, requires_grad=False)
    y_data = pnp.array(y_data, requires_grad=False)

    if initial_weights is not None:
        weights = pnp.array(initial_weights, requires_grad=True)
        print("Fine-tuning from existing weights...")
    else:
        weights = pnp.array(
            np.random.uniform(0, 2 * np.pi, (N_LAYERS, N_QUBITS, 3)),
            requires_grad=True
        )
        print("Training from scratch...")

    opt = qml.GradientDescentOptimizer(stepsize=lr)
    n_samples = len(X_data)
    n_batches_per_epoch = (n_samples + batch_size - 1) // batch_size
    print(f"Dataset: {n_samples} samples | Epochs: {epochs} | Batch size: {batch_size}")
    print(f"Batches per epoch: {n_batches_per_epoch} | Total batches: {n_batches_per_epoch * epochs}")
    print("-" * 60)

    history = []
    start_time = time.time()

    for epoch in range(1, epochs + 1):
        perm = np.random.permutation(n_samples)
        X_shuffled = X_data[perm]
        y_shuffled = y_data[perm]

        epoch_cost = 0.0
        n_batches = 0

        for i in range(0, n_samples, batch_size):
            Xb = X_shuffled[i:i+batch_size]
            yb = y_shuffled[i:i+batch_size]

            weights, batch_cost = opt.step_and_cost(
                lambda w, Xb=Xb, yb=yb: cost(w, Xb, yb),
                weights
            )
            epoch_cost += float(batch_cost)
            n_batches += 1

        avg_cost = epoch_cost / n_batches
        elapsed = time.time() - start_time
        eta = (elapsed / epoch) * (epochs - epoch)

        # Evaluate every 2 epochs or on first/last epoch
        if epoch % 2 == 0 or epoch == 1 or epoch == epochs:
            preds = predict_batch(weights, X_data[:120], threshold=RECALL_THRESHOLD)
            y_eval = pnp.where(y_data[:120] > 0, 1, 0)
            preds_01 = np.where(preds > 0, 1, 0)
            acc = accuracy_score(y_eval, preds_01)
            rec = recall_score(y_eval, preds_01, zero_division=0)
            print(f"Epoch {epoch:2d}/{epochs} | Cost: {avg_cost:.4f} | Acc: {acc:.2%} | Recall: {rec:.2%} | Time: {elapsed:.0f}s | ETA: {eta:.0f}s")
        else:
            print(f"Epoch {epoch:2d}/{epochs} | Cost: {avg_cost:.4f} | Time: {elapsed:.0f}s | ETA: {eta:.0f}s")

        history.append({"epoch": epoch, "cost": avg_cost})
        sys.stdout.flush()

    total_time = time.time() - start_time
    print(f"\nTraining complete in {total_time:.0f}s ({total_time/60:.1f} min)")
    return weights, history

# ============================================================
# 4. CONTINUOUS LEARNING / FINE-TUNING FUNCTION
# ============================================================
def continuous_learning(existing_weights_path, new_X, new_y, epochs=5, lr=0.005):
    print("\n" + "=" * 60)
    print("CONTINUOUS LEARNING: Fine-tuning on new patient data")
    print("=" * 60)
    with open(existing_weights_path, "r") as f:
        saved = json.load(f)
    old_weights = np.array(saved["weights"])
    new_y_qnn = 2 * new_y - 1

    print(f"Loaded weights trained on {saved.get('n_training_samples', '?')} samples")
    print(f"Fine-tuning on {len(new_X)} new patients")

    updated_weights, history = train_qnn(new_X, new_y_qnn, epochs=epochs, lr=lr, initial_weights=old_weights)

    output_path = existing_weights_path.replace(".json", "_updated.json")
    save_weights(updated_weights, output_path, n_samples=saved.get("n_training_samples", 0) + len(new_X))
    return updated_weights

# ============================================================
# 5. SAVE / LOAD UTILITIES
# ============================================================
def save_weights(weights, path, n_samples=0):
    data = {
        "weights": weights.tolist(),
        "n_qubits": N_QUBITS,
        "n_layers": N_LAYERS,
        "n_training_samples": n_samples,
        "recall_threshold": RECALL_THRESHOLD,
        "class_weight_positive": CLASS_WEIGHT_POS,
        "feature_names": ["BMI", "GenHlth", "Age", "HighBP", "HighChol", "Income", "PhysHlth", "Education"],
        "disease": "diabetes",
        "dataset": "CDC BRFSS 2015",
        "saved_at": time.strftime("%Y-%m-%d %H:%M:%S")
    }
    with open(path, "w") as f:
        json.dump(data, f, indent=2)
    size_bytes = os.path.getsize(path)
    n_params = N_LAYERS * N_QUBITS * 3
    print(f"Weights saved to {path} ({size_bytes} bytes, {n_params} trainable parameters)")
    print(f"  -> ZERO patient data stored. Fully privacy-preserving.")

# ============================================================
# 6. VISUALIZATION HELPERS
# ============================================================
def plot_training_curve(history):
    """Plot the cost curve to show the model learning over time."""
    epochs_list = [h["epoch"] for h in history]
    costs = [h["cost"] for h in history]

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(epochs_list, costs, 'b-o', linewidth=2, markersize=6)
    ax.set_xlabel("Epoch", fontsize=12)
    ax.set_ylabel("Weighted Cost (MSE)", fontsize=12)
    ax.set_title("Diabetes QNN Training Loss Curve", fontsize=14)
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig("diabetes_qnn_training_curve.png", dpi=150)
    plt.show()
    print("Training curve saved to diabetes_qnn_training_curve.png")

def plot_confusion_heatmap(y_true, y_pred, title_prefix="QNN"):
    """Generate side-by-side confusion matrix heatmaps."""
    cm = confusion_matrix(y_true, y_pred)
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Healthy', 'Diabetic'],
                yticklabels=['Healthy', 'Diabetic'], ax=axes[0],
                annot_kws={"size": 16})
    axes[0].set_title(f'{title_prefix} Confusion Matrix (Counts)', fontsize=13)
    axes[0].set_ylabel('Actual', fontsize=12)
    axes[0].set_xlabel('Predicted', fontsize=12)

    cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    sns.heatmap(cm_norm, annot=True, fmt='.1%', cmap='Reds',
                xticklabels=['Healthy', 'Diabetic'],
                yticklabels=['Healthy', 'Diabetic'], ax=axes[1],
                annot_kws={"size": 16})
    axes[1].set_title(f'{title_prefix} Per-Class Accuracy (Recall)', fontsize=13)
    axes[1].set_ylabel('Actual', fontsize=12)
    axes[1].set_xlabel('Predicted', fontsize=12)

    plt.tight_layout()
    fname = f"{title_prefix.lower().replace(' ', '_')}_confusion_matrix.png"
    plt.savefig(fname, dpi=150)
    plt.show()
    print(f"Heatmap saved to {fname}")

def print_metrics(y_true, y_pred, label=""):
    """Print all key metrics in a clean format."""
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)

    print(f"\n{'=' * 50}")
    print(f"  {label} METRICS")
    print(f"{'=' * 50}")
    print(f"  Accuracy:   {acc:.4f}")
    print(f"  Precision:  {prec:.4f}  (Of flagged, how many truly diabetic)")
    print(f"  Recall:     {rec:.4f}  (Of all diabetic, how many caught)  <-- KEY METRIC")
    print(f"  F1-Score:   {f1:.4f}")
    print(f"{'=' * 50}")
    return acc, prec, rec, f1

# ============================================================
# 7. THRESHOLD SEARCH â€” Find optimal recall-maximizing cutoff
# ============================================================
def find_best_recall_threshold(weights, X_val, y_val_01, thresholds=None):
    """Search over thresholds to find the one that maximizes recall while keeping F1 > 0.30."""
    if thresholds is None:
        thresholds = np.arange(-0.8, 0.4, 0.05)
    
    print("\nSearching for optimal recall threshold...")
    best_recall = 0
    best_threshold = 0.0
    best_f1 = 0
    
    # Get raw circuit outputs once (expensive part)
    raw_outputs = np.array([float(qnn_circuit(weights, x)) for x in X_val])
    
    for t in thresholds:
        preds_qnn = np.where(raw_outputs >= t, 1, -1)
        preds_01 = np.where(preds_qnn > 0, 1, 0)
        rec = recall_score(y_val_01, preds_01, zero_division=0)
        f1 = f1_score(y_val_01, preds_01, zero_division=0)
        prec = precision_score(y_val_01, preds_01, zero_division=0)
        
        # Prioritize F1 score and balanced accuracy, requiring at least 60% recall
        if f1 > best_f1 and rec >= 0.60:
            best_recall = rec
            best_threshold = t
            best_f1 = f1
        
        if abs(t - RECALL_THRESHOLD) < 0.01 or abs(t) < 0.01 or rec > 0.85:
            print(f"  threshold={t:+.2f} | Recall={rec:.2%} | Precision={prec:.2%} | F1={f1:.4f}")
    
    print(f"\n  >>> Best threshold for recall: {best_threshold:+.2f} (Recall={best_recall:.2%}, F1={best_f1:.4f})")
    return best_threshold

# ============================================================
# 8. MAIN EXECUTION
# ============================================================
if __name__ == "__main__":

    # ---- PHASE 1: TRAIN ----
    print("\n[PHASE 1] Training QNN on balanced diabetes dataset...")
    print(f"Expected runtime: ~45-60 mins on Colab (8 qubits, 3 layers, 40 epochs, 300 samples)")
    trained_weights, train_history = train_qnn(X_balanced, y_balanced)

    os.makedirs("models", exist_ok=True)
    save_weights(trained_weights, "models/diabetes_qnn_weights.json", n_samples=len(X_balanced))

    # Plot training loss curve
    plot_training_curve(train_history)

    # ---- PHASE 2: FIND OPTIMAL THRESHOLD ----
    print("\n[PHASE 2] Searching for optimal recall threshold on test subset...")
    search_n = min(200, len(X_test))
    search_idx = np.random.choice(len(X_test), search_n, replace=False)
    optimal_threshold = find_best_recall_threshold(
        trained_weights, X_test[search_idx], y_test[search_idx]
    )

    # ---- PHASE 3: EVALUATE ----
    print("\n[PHASE 3] Evaluating on unseen test set...")
    eval_n = min(300, len(X_test))
    eval_idx = np.random.choice(len(X_test), eval_n, replace=False)
    X_eval = X_test[eval_idx]
    y_eval = y_test[eval_idx]

    print(f"Evaluating on {eval_n} test patients...")
    print(f"Using recall-maximizing threshold: {optimal_threshold:+.2f}")

    y_pred_qnn = predict_batch(trained_weights, X_eval, threshold=optimal_threshold)
    y_pred_01 = np.where(y_pred_qnn > 0, 1, 0)

    acc, prec, rec, f1 = print_metrics(y_eval, y_pred_01, label="QNN Diabetes (Recall-Optimized)")
    plot_confusion_heatmap(y_eval, y_pred_01, title_prefix="Diabetes QNN Recall-Optimized")

    print("\n[COMPARISON] With neutral threshold (0.0):")
    y_pred_default = predict_batch(trained_weights, X_eval, threshold=0.0)
    y_pred_default_01 = np.where(y_pred_default > 0, 1, 0)
    print_metrics(y_eval, y_pred_default_01, label="QNN Diabetes (Neutral Threshold)")

    # ---- PHASE 4: CONTINUOUS LEARNING DEMO ----
    print("\n[PHASE 4] Demonstrating Continuous Learning (40 new patients)...")
    new_n = min(40, len(X_test))
    new_idx = np.random.choice(len(X_test), new_n, replace=False)
    new_X, new_y = X_test[new_idx], y_test[new_idx]

    updated_weights = continuous_learning("models/diabetes_qnn_weights.json", new_X, new_y, epochs=3, lr=0.005)

    print("\n" + "=" * 60)
    print("ALL DONE.")
    print("=" * 60)





## Download All Results


In [ ]:

from google.colab import files
import glob

for f in glob.glob("models/diabetes*.json") + glob.glob("diabetes*.png"):
    files.download(f)
print("Done!")
